In [2]:
# Using huggingface_hub
from huggingface_hub import hf_hub_download

# Download the recommended latest model
model_path = hf_hub_download(repo_id="wanglab/MedSAM2", filename="MedSAM2_latest.pt")

# Or download a specific fine-tuned model
heart_us_model_path = hf_hub_download(repo_id="wanglab/MedSAM2", filename="MedSAM2_US_Heart.pt")
liver_model_path = hf_hub_download(repo_id="wanglab/MedSAM2", filename="MedSAM2_MRI_LiverLesion.pt")


In [3]:
from huggingface_hub import hf_hub_download
import os

# Create checkpoints directory if it doesn't exist
os.makedirs("checkpoints", exist_ok=True)

# List of model filenames
model_files = [
    "MedSAM2_2411.pt",
    "MedSAM2_US_Heart.pt",
    "MedSAM2_MRI_LiverLesion.pt",
    "MedSAM2_CTLesion.pt",
    "MedSAM2_latest.pt"
]

# Download all models
for model_file in model_files:
    local_path = os.path.join("checkpoints", model_file)
    hf_hub_download(
        repo_id="wanglab/MedSAM2",
        filename=model_file,
        local_dir="checkpoints",
        local_dir_use_symlinks=False
    )
    print(f"Downloaded {model_file} to {local_path}")


c:\Users\HP\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:982: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(


Downloaded MedSAM2_2411.pt to checkpoints\MedSAM2_2411.pt
Downloaded MedSAM2_US_Heart.pt to checkpoints\MedSAM2_US_Heart.pt
Downloaded MedSAM2_MRI_LiverLesion.pt to checkpoints\MedSAM2_MRI_LiverLesion.pt
Downloaded MedSAM2_CTLesion.pt to checkpoints\MedSAM2_CTLesion.pt
Downloaded MedSAM2_latest.pt to checkpoints\MedSAM2_latest.pt


In [2]:
import torch
import os
import sam2
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

# --- 1. Setup ---
device = "cuda" if torch.cuda.is_available() else "cpu"
checkpoint_path = "checkpoints/MedSAM2_latest.pt"

# Find the installed config directory
sam2_dir = os.path.dirname(sam2.__file__)
# Common paths where pip installs configs
possible_config_dirs = [
    os.path.join(sam2_dir, "configs", "sam2"),
    os.path.join(sam2_dir, "configs"),
    os.path.join(sam2_dir, "..", "sam2_configs")
]

config_root = None
for d in possible_config_dirs:
    if os.path.exists(d):
        config_root = d
        break

if not config_root:
    raise FileNotFoundError("Could not find 'sam2_configs' directory. Reinstall sam2.")

# --- 2. Define Architectures to Try ---
# We order them from most likely to least likely for MedSAM2
configs_to_try = [
    "sam2_hiera_b+.yaml",  # Base Plus (Most likely for 768 dim)
    "sam2_hiera_l.yaml",   # Large
    "sam2_hiera_s.yaml",   # Small
    "sam2_hiera_t.yaml",   # Tiny
    "sam2_hiera_b.yaml",   # Base
]

# --- 3. Auto-Discovery Loop ---
medsam_model = None
selected_config = None

print(f"Probing architectures for {checkpoint_path}...")

for config_name in configs_to_try:
    cfg_path = os.path.join(config_root, config_name)
    if not os.path.exists(cfg_path):
        continue
        
    print(f"\nTesting config: {config_name} ...")
    
    try:
        # Build model skeleton
        temp_model = build_sam2(cfg_path, checkpoint_path=None, device=device)
        
        # Load weights
        ckpt = torch.load(checkpoint_path, map_location="cpu")
        state_dict = ckpt["model"] if "model" in ckpt else ckpt
        
        # strict=True forces a size check immediately
        # We use strict=True inside a try/catch to detect size mismatches
        # However, SAM2 often has missing keys (heads), so we manually check for size mismatch errors
        try:
            temp_model.load_state_dict(state_dict, strict=True)
        except RuntimeError as e:
            if "size mismatch" in str(e):
                print(f" -> Mismatch detected. (Not this one)")
                continue # Try next config
            elif "Missing key" in str(e):
                # Missing keys are fine (usually decoder heads), as long as sizes match!
                # If we get here without a size mismatch error, we found it.
                pass
        
        # If we passed the size check, this is the one!
        # Reload with strict=False to actually load it
        temp_model.load_state_dict(state_dict, strict=False)
        medsam_model = temp_model
        selected_config = config_name
        print(f" -> MATCH FOUND! The correct config is {config_name}")
        break
        
    except Exception as e:
        print(f" -> Error testing {config_name}: {e}")

# --- 4. Final Result ---
if medsam_model:
    print(f"\nSUCCESS: Model loaded using {selected_config}")
    predictor = SAM2ImagePredictor(medsam_model)
    print("Predictor is ready.")
else:
    print("\nFAILURE: No matching configuration found. Are you sure this is a SAM 2 checkpoint?")

Probing architectures for checkpoints/MedSAM2_latest.pt...

Testing config: sam2_hiera_b+.yaml ...
 -> Mismatch detected. (Not this one)

Testing config: sam2_hiera_l.yaml ...
 -> Mismatch detected. (Not this one)

Testing config: sam2_hiera_s.yaml ...
 -> Mismatch detected. (Not this one)

Testing config: sam2_hiera_t.yaml ...
 -> MATCH FOUND! The correct config is sam2_hiera_t.yaml

SUCCESS: Model loaded using sam2_hiera_t.yaml
Predictor is ready.


In [15]:
import os
import glob
import torch
import numpy as np
import nibabel as nib
import plotly.graph_objects as go
from skimage import measure
from tqdm import tqdm # Use notebook version of tqdm for better bars
import json
import sam2
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

# --- CONFIGURATION ---
class Config:
    # INPUT: Path to your "Training Batch 2" folder containing volume-X.nii files
    DATA_DIR = r"D:\mini project\Training_Batch2\media\nas\01_Datasets\CT\LITS\Training Batch 2"
    
    # MODEL: Path to your checkpoint file
    CHECKPOINT_PATH = "checkpoints/MedSAM2_latest.pt"
    
    # OUTPUT: Folder where results will be saved
    OUTPUT_DIR = "results"
    
    # PARAMETERS
    CONFIDENCE_THRESHOLD = 0.5
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Create output folder
os.makedirs(Config.OUTPUT_DIR, exist_ok=True)
print(f"Configuration loaded. Device: {Config.DEVICE}")

Configuration loaded. Device: cuda


In [23]:
"""
MedSAM-2 Final Pipeline (Colored Visualization)
===============================================
1. Robustly loads MedSAM-2 (Fixes architecture/size mismatch).
2. Normalizes CT contrast for the Liver (Fixes black window).
3. Uses Ground Truth to prompt the model (Guarantees detection).
4. OUTPUT 1: A 3D interactive file where the tumor is RED (Matches your requested style).
5. OUTPUT 2: A 2D snapshot where the tumor is GREEN (For instant verification).
"""

import os
import glob
import torch
import numpy as np
import nibabel as nib
import cv2  # OpenCV for 2D image processing
import plotly.graph_objects as go
from skimage import measure
import sam2
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

# --- CONFIGURATION ---
class Config:
    # FOLDER containing your .nii files
    DATA_DIR = r"D:\mini project\Training_Batch2\media\nas\01_Datasets\CT\LITS\Training Batch 2"
    CHECKPOINT_PATH = "checkpoints/MedSAM2_latest.pt"
    OUTPUT_DIR = "results_colored"
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

os.makedirs(Config.OUTPUT_DIR, exist_ok=True)

# ==========================================
# 1. ROBUST MODEL LOADER
# ==========================================
def load_medsam_model(checkpoint, device):
    print(f"\n[1/4] Loading MedSAM-2 on {device}...")
    sam2_dir = os.path.dirname(sam2.__file__)
    roots = [
        os.path.join(sam2_dir, "configs", "sam2.1"),
        os.path.join(sam2_dir, "configs", "sam2"),
        os.path.join(sam2_dir, "configs"),
        os.path.join(sam2_dir, "..", "sam2_configs"),
    ]
    config_root = next((d for d in roots if os.path.exists(d)), None)
    if not config_root: raise FileNotFoundError("SAM2 configs not found.")
    
    configs = glob.glob(os.path.join(config_root, "*.yaml"))
    state = torch.load(checkpoint, map_location="cpu")
    if "model" in state: state = state["model"]

    for cfg in configs:
        if "common" in cfg: continue
        try:
            model = build_sam2(cfg, checkpoint_path=None, device=device)
            # Strict=False allows loading research weights into standard architecture
            model.load_state_dict(state, strict=False)
            print(f"✓ Success: Loaded config {os.path.basename(cfg)}")
            return SAM2ImagePredictor(model)
        except: continue
    raise RuntimeError("Model architecture mismatch. Check your checkpoint.")

# ==========================================
# 2. PREPROCESSING (Liver Window)
# ==========================================
def get_liver_window(nii_path):
    nii = nib.load(nii_path)
    data = nii.get_fdata()
    # Clip to [-135, 215] (Standard Liver Window)
    # This ensures the liver and tumor are visible grey, not black/white noise
    data = np.clip(data, -135, 215)
    # Normalize 0-255
    data = ((data + 135) / (215 + 135) * 255.0).astype(np.uint8)
    return data, nii.affine

def get_oracle_prompt(nii_path):
    """Finds the true tumor center from the segmentation file."""
    seg_path = nii_path.replace("volume", "segmentation")
    if not os.path.exists(seg_path): return None, None
    
    seg = nib.load(seg_path).get_fdata()
    coords = np.argwhere(seg > 0)
    if len(coords) == 0: return None, None
    
    # Find slice with most tumor pixels (Best prompt location)
    z_counts = np.bincount(coords[:, 2])
    best_z = np.argmax(z_counts)
    
    # Get bounding box [x_min, y_min, x_max, y_max]
    slice_pts = coords[coords[:, 2] == best_z]
    y_min, x_min = slice_pts[:, 0].min(), slice_pts[:, 1].min()
    y_max, x_max = slice_pts[:, 0].max(), slice_pts[:, 1].max()
    
    return best_z, [x_min, y_min, x_max, y_max]

# ==========================================
# 3. VISUALIZATION FUNCTIONS (Red 3D & Green 2D)
# ==========================================
def save_3d_red_tumor(mask, filename):
    """Saves interactive 3D plot with RED tumor."""
    print("   -> Generating 3D Plot...")
    if mask.sum() == 0: return
    
    # Marching Cubes Algorithm to create 3D mesh from mask
    verts, faces, _, _ = measure.marching_cubes(mask, level=0.5)
    
    fig = go.Figure(data=[go.Mesh3d(
        x=verts[:, 0], y=verts[:, 1], z=verts[:, 2],
        i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
        opacity=0.8,       # High opacity for solid look
        color='red',       # <--- FORCE COLOR TO RED
        name='Tumor'
    )])
    fig.update_layout(title=f"Tumor 3D: {filename}", scene=dict(aspectmode='data'))
    
    out_path = os.path.join(Config.OUTPUT_DIR, f"{filename}_3d.html")
    fig.write_html(out_path)
    print(f"   ✓ Saved 3D HTML: {out_path}")

def save_2d_green_snapshot(volume, mask, z_index, filename):
    """Saves a JPG of the slice with GREEN tumor overlay."""
    img = volume[:, :, z_index]
    m = mask[:, :, z_index]
    
    # Make RGB
    img_rgb = np.stack([img]*3, axis=-1)
    
    # Create Green Overlay
    overlay = np.zeros_like(img_rgb)
    overlay[m == 1] = [0, 255, 0] # Green
    
    # Blend: 70% Original, 30% Green
    final_img = cv2.addWeighted(img_rgb, 0.7, overlay, 0.3, 0)
    
    # Draw Red Contour for extra visibility
    contours, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(final_img, contours, -1, (0, 0, 255), 2)
    
    out_path = os.path.join(Config.OUTPUT_DIR, f"{filename}_slice{z_index}_check.jpg")
    cv2.imwrite(out_path, final_img)
    print(f"   ✓ Saved 2D Snapshot: {out_path}")

# ==========================================
# 4. MAIN LOOP
# ==========================================
def main():
    predictor = load_medsam_model(Config.CHECKPOINT_PATH, Config.DEVICE)
    
    # Find volumes
    vol_files = glob.glob(os.path.join(Config.DATA_DIR, "volume-*.nii"))
    print(f"\n[2/4] Found {len(vol_files)} volumes.")
    
    for i, nii_path in enumerate(vol_files):
        name = os.path.basename(nii_path).replace(".nii", "")
        print(f"\n--- Processing {name} ({i+1}/{len(vol_files)}) ---")
        
        # 1. Get Prompt
        target_z, bbox = get_oracle_prompt(nii_path)
        if target_z is None:
            print("   [!] No segmentation/tumor found. Skipping.")
            continue
            
        print(f"   Prompting Slice {target_z} with Box {bbox}")
        
        # 2. Load Volume
        vol, _ = get_liver_window(nii_path)
        
        # 3. Run Inference (Chunk around the tumor to save time)
        mask_3d = np.zeros_like(vol, dtype=np.uint8)
        start_z = max(0, target_z - 20)
        end_z = min(vol.shape[2], target_z + 20)
        
        print(f"   Segmenting slices {start_z}-{end_z}...")
        for z in range(start_z, end_z):
            img_slice = np.stack([vol[:, :, z]]*3, axis=-1)
            predictor.set_image(img_slice)
            masks, scores, _ = predictor.predict(
                box=np.array([bbox]), multimask_output=False
            )
            if scores[0] > 0.5:
                mask_3d[:, :, z] = masks[0]

        # 4. SAVE COLORED OUTPUTS
        if mask_3d.sum() > 0:
            # Save 3D Red Plot
            save_3d_red_tumor(mask_3d, name)
            # Save 2D Green Snapshot
            save_2d_green_snapshot(vol, mask_3d, target_z, name)
        else:
            print("   [!] Model returned empty mask.")

    print(f"\n[Done] Results saved in folder: {Config.OUTPUT_DIR}")

if __name__ == "__main__":
    main()


[1/4] Loading MedSAM-2 on cuda...
✓ Success: Loaded config sam2.1_hiera_t.yaml

[2/4] Found 103 volumes.

--- Processing volume-100 (1/103) ---
   Prompting Slice 557 with Box [np.int64(97), np.int64(42), np.int64(368), np.int64(282)]
   Segmenting slices 537-577...
   -> Generating 3D Plot...
   ✓ Saved 3D HTML: results_colored\volume-100_3d.html
   ✓ Saved 2D Snapshot: results_colored\volume-100_slice557_check.jpg

--- Processing volume-101 (2/103) ---
   Prompting Slice 557 with Box [np.int64(118), np.int64(32), np.int64(357), np.int64(324)]
   Segmenting slices 537-577...
   -> Generating 3D Plot...
   ✓ Saved 3D HTML: results_colored\volume-101_3d.html
   ✓ Saved 2D Snapshot: results_colored\volume-101_slice557_check.jpg

--- Processing volume-102 (3/103) ---
   Prompting Slice 575 with Box [np.int64(93), np.int64(58), np.int64(344), np.int64(376)]
   Segmenting slices 555-595...
   -> Generating 3D Plot...
   ✓ Saved 3D HTML: results_colored\volume-102_3d.html
   ✓ Saved 2D Snap

KeyboardInterrupt: 